# PriorityLab — M0 EDA & Preprocessing

**Goal:** load the synthetic email dataset, validate schema/labels, audit quality, and decide go/no-go for Milestone 1 (TF-IDF + Logistic Regression).

**Rules:** subject + body only as future model inputs; `category` / `action` / `source_round` are audit-only. No model training in this notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset

ROOT = Path("..").resolve()
FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    "blue": "#4285F4",
    "red": "#EA4335",
    "yellow": "#FBBC04",
    "green": "#34A853",
}
PRIORITY_TO_IMPORTANT = {"urgent": 1, "high": 1, "medium": 0, "low": 0}
REQUIRED = {"subject", "body", "priority", "category", "action", "source_round"}

pd.set_option("display.max_colwidth", 120)
print("FIG_DIR:", FIG_DIR)

## 1. Load dataset

In [ ]:
raw = load_dataset("weijianzhg/email-triage-action-seed", split="train")
df = raw.to_pandas()
missing_cols = REQUIRED - set(df.columns)
assert not missing_cols, f"Missing columns: {missing_cols}"
print("shape:", df.shape)
print("columns:", list(df.columns))
df.head(3)

## 2. Priority distribution + binary target

In [ ]:
print("priority value_counts (raw):")
print(df["priority"].value_counts(dropna=False))

df = df.copy()
df["priority_norm"] = df["priority"].astype("string").str.strip().str.lower()
unknown = set(df["priority_norm"].dropna().unique()) - set(PRIORITY_TO_IMPORTANT)
assert not unknown and df["priority_norm"].notna().all(), f"Bad priority labels: {unknown}"
df["important"] = df["priority_norm"].map(PRIORITY_TO_IMPORTANT).astype(int)
assert df["important"].nunique() == 2, "Both binary classes must be present"

print("\nimportant value_counts:")
print(df["important"].value_counts().rename({1: "important", 0: "not_important"}))
print("\nprevalence important:", round(df["important"].mean(), 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

prio_order = [p for p in ["urgent", "high", "medium", "low"] if p in set(df["priority_norm"])]
prio_counts = df["priority_norm"].value_counts().reindex(prio_order)
prio_colors = [COLORS["red"], COLORS["yellow"], COLORS["blue"], COLORS["green"]][: len(prio_order)]
axes[0].bar(prio_counts.index.astype(str), prio_counts.values, color=prio_colors)
axes[0].set_title("Source priority")
axes[0].set_ylabel("Count")

imp_counts = df["important"].value_counts().reindex([1, 0])
axes[1].bar(["important", "not_important"], imp_counts.values, color=[COLORS["blue"], COLORS["green"]])
axes[1].set_title("Derived binary target")

fig.tight_layout()
out = FIG_DIR / "m0_label_distributions.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("saved:", out)
plt.show()

## 3. Quality checks (missing, empty, duplicates)

In [ ]:
df["subject_clean"] = df["subject"].fillna("").astype(str).str.strip()
df["body_clean"] = df["body"].fillna("").astype(str).str.strip()
df["text"] = "subject: " + df["subject_clean"] + " body: " + df["body_clean"]

print("null counts:\n", df[list(REQUIRED)].isna().sum())
print("\nempty subject:", (df["subject_clean"] == "").sum())
print("empty body:", (df["body_clean"] == "").sum())
print("empty both:", ((df["subject_clean"] == "") & (df["body_clean"] == "")).sum())

dup_exact = df.duplicated(subset=["text"], keep=False).sum()
dup_pairs = df.duplicated(subset=["text"], keep="first").sum()
print("rows in exact-duplicate text groups:", int(dup_exact))
print("exact duplicate rows if dropped (keep first):", int(dup_pairs))

# Contradictory duplicates: same text, different important label
grp = df.groupby("text")["important"].nunique()
contradict = int((grp > 1).sum())
print("texts with contradictory important labels:", contradict)

## 4. Length + category / action audit

In [ ]:
df["subject_len"] = df["subject_clean"].str.len()
df["body_len"] = df["body_clean"].str.len()
print(df[["subject_len", "body_len"]].describe().round(1))

print("\ncategory value_counts:")
print(df["category"].value_counts(dropna=False).head(20))
print("\naction value_counts:")
print(df["action"].value_counts(dropna=False).head(20))

print("\npriority × action (top):")
print(pd.crosstab(df["priority_norm"], df["action"]).head())

if "source_round" in df.columns:
    print("\nsource_round value_counts:")
    print(df["source_round"].value_counts(dropna=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(df["subject_len"], bins=30, color=COLORS["blue"], edgecolor="white")
axes[0].set_title("Subject length (chars)")
axes[0].set_xlabel("Characters")
axes[1].hist(df["body_len"], bins=30, color=COLORS["green"], edgecolor="white")
axes[1].set_title("Body length (chars)")
axes[1].set_xlabel("Characters")
fig.tight_layout()
out = FIG_DIR / "m0_text_lengths.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("saved:", out)
plt.show()

## 5. Manual sample review

In [ ]:
for label, name in [(1, "important"), (0, "not_important")]:
    print("=" * 60)
    print(name)
    sample = df.loc[df["important"] == label, ["subject_clean", "body_clean", "priority_norm", "category", "action"]].sample(
        5, random_state=42
    )
    display(sample)

## 6. Go / no-go for Milestone 1

Fill after running cells above:

- [ ] Both binary classes present with usable support
- [ ] Schema matches expected fields
- [ ] Duplicate / contradiction risk documented
- [ ] Synthetic bias acknowledged (LLM-generated English emails)
- [ ] **Decision:** proceed to stratified split + TF-IDF baseline **only if** labels are usable

**Next (still M0/prep):** choose split strategy (prefer group-aware if many near-duplicates), export a split manifest, then start M1 training code.